# ETL Spark — Bronze → Silver → Gold (Tesouro Direto)

Lê os dados brutos (JSON) entregues pelo Kafka Connect no **S3** (Bronze) e gera
**Silver** (descarte de inválidos, dedup por chave de negócio, datas, normalização)
e **Gold** (agregações por título), gravando Parquet de volta no S3.


In [9]:
import sys; print(sys.executable)

/opt/conda/bin/python


In [10]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 pyspark-shell")

In [11]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (col, from_unixtime, to_date, avg, count,
    round as sround, row_number, min as smin, max as smax)

spark = (SparkSession.builder.appName("ETL Tesouro - Bronze/Silver/Gold")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
CHAVE = ["Tipo", "Data_Vencimento", "Data_Base"]   # chave de negócio

## Bronze — leitura dos JSON brutos no S3

In [12]:
BUCKET_IPCA = "s3a://my-bucket-cabxavier-01"
BUCKET_PRE  = "s3a://my-bucket-cabxavier-02"
df_bronze_ipca = spark.read.json(f"{BUCKET_IPCA}/raw-data/kafka/postgres-dadostesouroipca/")
df_bronze_pre  = spark.read.json(f"{BUCKET_PRE}/raw-data/kafka/postgres-dadostesouropre/")
print("Bronze IPCA:", df_bronze_ipca.count(), "| Bronze PRE:", df_bronze_pre.count())
df_bronze_ipca.printSchema(); df_bronze_ipca.show(5, truncate=False)

Bronze IPCA: 500 | Bronze PRE: 500
root
 |-- CompraManha: double (nullable = true)
 |-- Data_Base: long (nullable = true)
 |-- Data_Vencimento: long (nullable = true)
 |-- PUBaseManha: double (nullable = true)
 |-- PUCompraManha: double (nullable = true)
 |-- PUVendaManha: double (nullable = true)
 |-- Tipo: string (nullable = true)
 |-- VendaManha: double (nullable = true)
 |-- dt_update: long (nullable = true)
 |-- partition: integer (nullable = true)

+-----------+-------------+---------------+-----------+-------------+------------+----------------------------------+----------+-------------+---------+
|CompraManha|Data_Base    |Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo                              |VendaManha|dt_update    |partition|
+-----------+-------------+---------------+-----------+-------------+------------+----------------------------------+----------+-------------+---------+
|10.04      |1144281600000|1179187200000  |1567.32    |1568.45      |1568.15     |

## Silver — qualidade e transformação
1. Descarta registros inválidos (PU de compra/venda ausente).
2. Deduplica por chave de negócio (Tipo+Vencimento+Base), mantendo o `dt_update` mais recente.
3. Converte timestamps (epoch ms → data) e normaliza.

In [13]:
def to_silver(df):
    df_valid = df.filter(col("PUCompraManha").isNotNull() & col("PUVendaManha").isNotNull())
    w = Window.partitionBy(*CHAVE).orderBy(col("dt_update").desc())
    df_dedup = (df_valid.withColumn("_rn", row_number().over(w))
                        .filter(col("_rn") == 1).drop("_rn"))
    return (df_dedup
        .withColumn("Data_Vencimento", to_date(from_unixtime(col("Data_Vencimento")/1000)))
        .withColumn("Data_Base",       to_date(from_unixtime(col("Data_Base")/1000)))
        .withColumn("dt_update",        from_unixtime(col("dt_update")/1000, "yyyy-MM-dd HH:mm:ss")))

df_silver_ipca = to_silver(df_bronze_ipca); df_silver_pre = to_silver(df_bronze_pre)
print("Silver IPCA:", df_silver_ipca.count(), "| Silver PRE:", df_silver_pre.count())
df_silver_ipca.orderBy("Data_Base").show(5, truncate=False)
df_silver_ipca.write.mode("overwrite").parquet(f"{BUCKET_IPCA}/processed-data/ipca/silver/")
df_silver_pre.write.mode("overwrite").parquet(f"{BUCKET_PRE}/processed-data/pre/silver/")

Silver IPCA: 500 | Silver PRE: 500
+-----------+----------+---------------+-----------+-------------+------------+----------------------------------+----------+-------------------+---------+
|CompraManha|Data_Base |Data_Vencimento|PUBaseManha|PUCompraManha|PUVendaManha|Tipo                              |VendaManha|dt_update          |partition|
+-----------+----------+---------------+-----------+-------------+------------+----------------------------------+----------+-------------------+---------+
|8.78       |2004-12-31|2009-05-15     |1348.51    |1351.84      |1349.94     |Tesouro IPCA+ com Juros Semestrais|8.82      |2026-06-12 13:04:58|0        |
|8.76       |2004-12-31|2015-05-15     |1219.16    |1227.19      |1220.45     |Tesouro IPCA+ com Juros Semestrais|8.84      |2026-06-12 13:04:58|0        |
|8.71       |2004-12-31|2006-08-15     |1454.56    |1456.5       |1456.09     |Tesouro IPCA+ com Juros Semestrais|8.73      |2026-06-12 13:04:58|0        |
|8.83       |2004-12-31|2024-

## Gold — agregações analíticas por título

In [14]:
def to_gold(df):
    return (df.groupBy("Tipo").agg(
        sround(avg("PUCompraManha"),2).alias("Media_PUCompraManha"),
        sround(avg("PUVendaManha"),2).alias("Media_PUVendaManha"),
        sround(avg("CompraManha"),4).alias("Media_TaxaCompra"),
        sround(smin("PUCompraManha"),2).alias("Min_PUCompra"),
        sround(smax("PUCompraManha"),2).alias("Max_PUCompra"),
        count("*").alias("Total_Registros")))
df_gold_ipca = to_gold(df_silver_ipca); df_gold_pre = to_gold(df_silver_pre)
df_gold_ipca.show(truncate=False); df_gold_pre.show(truncate=False)
df_gold_ipca.write.mode("overwrite").parquet(f"{BUCKET_IPCA}/analytics/ipca/gold/")
df_gold_pre.write.mode("overwrite").parquet(f"{BUCKET_PRE}/analytics/pre/gold/")

+----------------------------------+-------------------+------------------+----------------+------------+------------+---------------+
|Tipo                              |Media_PUCompraManha|Media_PUVendaManha|Media_TaxaCompra|Min_PUCompra|Max_PUCompra|Total_Registros|
+----------------------------------+-------------------+------------------+----------------+------------+------------+---------------+
|Tesouro IPCA+                     |612.6              |607.7             |7.6588          |404.02      |835.99      |66             |
|Tesouro IPCA+ com Juros Semestrais|1383.67            |1378.66           |9.085           |1018.52     |1592.49     |434            |
+----------------------------------+-------------------+------------------+----------------+------------+------------+---------------+

+--------------------------------------+-------------------+------------------+----------------+------------+------------+---------------+
|Tipo                                  |Media_PUCo

## Validação (Spark SQL)

In [15]:
df_gold_ipca.unionByName(df_gold_pre).createOrReplaceTempView("gold_tesouro")
spark.sql("SELECT Tipo, Total_Registros, Media_PUCompraManha FROM gold_tesouro "
          "ORDER BY Media_PUCompraManha DESC").show(truncate=False)

+--------------------------------------+---------------+-------------------+
|Tipo                                  |Total_Registros|Media_PUCompraManha|
+--------------------------------------+---------------+-------------------+
|Tesouro IPCA+ com Juros Semestrais    |434            |1383.67            |
|Tesouro Prefixado com Juros Semestrais|115            |892.45             |
|Tesouro Prefixado                     |385            |872.22             |
|Tesouro IPCA+                         |66             |612.6              |
+--------------------------------------+---------------+-------------------+

